# Making word embeddings with FastText

## Stuff from assignment 1 to load dataset

In [ ]:
from collections import Counter
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import random
from sympy import true

import part1 # to use code from tutorial
import torch

def set_seed(seed):
    """Seeds Python, NumPy (also used by sklearn's saga solver) and PyTorch (weight init, shuffling, dropout)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:
# splitting data into train and test sets and folds for cross-validation
LABEL_COL = "Party_orientation"
SAMPLE_FRAC = 1.0  # fraction of the data to use, 1.0 for all of it

# get the dataframe: one dataframe with tokens, lemmas, sentences and embeddings,
# so all three representations are trained and evaluated on the same rows
df = pd.read_pickle("./merged_embeddings_sampled.pkl")
df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)

# train test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df[LABEL_COL])
train_df = train_df.reset_index(drop=True)

# labels and settings
label_names = sorted(df[LABEL_COL].unique())
outcome_space = part1.clslib.NamedClasses(label_names)
# repeated cross-validation: each seed gives its own fold split and model initialisation
# (fixed in advance; more runs give the Bayesian correlated t-test more paired differences to work with)
my_seeds = [0, 1, 2]
num_folds = 5

## Loading fasttext

In [ ]:
from gensim.models import KeyedVectors

kv = KeyedVectors.load_word2vec_format("cc.hu.300.vec", limit=500_000)
print(len(kv.key_to_index), kv.vector_size)

In [ ]:
from collections import Counter

TOKEN_COL, LEMMA_COL = "tokens", "lemmas"

def to_list(x):
    return x.split() if isinstance(x, str) else list(x)

pair_counts, token_counts, lemma_counts = Counter(), Counter(), Counter()
for toks, lems in zip(train_df[TOKEN_COL], train_df[LEMMA_COL]):
    toks, lems = to_list(toks), to_list(lems)
    token_counts.update(toks)
    lemma_counts.update(lems)
    pair_counts.update(zip(toks, lems))
print("unique pairs:", len(pair_counts))

## Check coverage

In [ ]:
def resolve(w, kv):
    """Return the key in kv that matches the lowercase word w, trying common casings, or None."""
    for cand in (w, w.capitalize(), w.upper()):
        if cand in kv.key_to_index:
            return cand
    return None

def coverage(counts, kv):
    tokens = types = 0
    for w, n in counts.items():
        if resolve(w, kv) is not None:
            tokens += n
            types += 1
    return tokens / sum(counts.values()), types / len(counts)

for name, counts in [("tokens", token_counts), ("lemmas", lemma_counts)]:
    t, ty = coverage(counts, kv)
    print(f"{name:7} token coverage {t:.3f},  type coverage {ty:.3f}")

## And missing words

In [ ]:
def top_missing(counts, kv, n=30):
    miss = [(w, c) for w, c in counts.most_common() if resolve(w, kv) is None]
    return miss[:n]

print("tokens:", top_missing(token_counts, kv))
print("lemmas:", top_missing(lemma_counts, kv))

## Checking missing words with higher limit

In [ ]:
kv_full = KeyedVectors.load_word2vec_format("cc.hu.300.vec", limit=2_000_000)
for name, counts in [("tokens", token_counts), ("lemmas", lemma_counts)]:
    t, ty = coverage(counts, kv_full)
    print(f"{name:7} token coverage {t:.3f},   type coverage {ty:.3f}")

## Getting suffixes and counts

In [ ]:
# ház -> házban works, we get -ban, but for example kéz -> kezet does not
def get_suffix(tok, lem):
    if tok != lem and tok.startswith(lem):
        return tok[len(lem):]
    return None

suffix_info = {}   # list of (token_key, lemma_key, count)
for (tok, lem), n in pair_counts.items():
    s = get_suffix(tok, lem)
    if not s:
        continue
    tk, lk = resolve(tok, kv_full), resolve(lem, kv_full)
    if tk and lk:
        suffix_info.setdefault(s, []).append((tk, lk, n))

# tok and lem are the words as in our df (lowercase)
# tk, lk are the same words but as in the model
# suffix_info looks like: {"ban": [("házban", "ház", 12), ...], "t": [...], ...}
MIN_PAIRS = 50
MAX_SUF = 30
suffixes = sorted([s for s, v in suffix_info.items() if len(v) >= MIN_PAIRS],
                  key=lambda s: -len(suffix_info[s]))[:MAX_SUF]
print({s: len(suffix_info[s]) for s in suffixes})

# suffixes can be ambiguous tho, t is accusative or past tense for example

## Cos similarity and offset consistency

In [ ]:
# For a pair like (házban, ház), the offset is the vector token − lemma. 
# It’s the difference that the suffix -ban causes in the embedding space. 
# If -ban means the same thing in every word, the offsets for házban − ház, 
# asztalban − asztal and so on should all point in a similar direction.

def unit(M): return M / np.linalg.norm(M, axis=1, keepdims=True)

def token_lemma_cos(T, L):
    """Cosine between each token and its own lemma, averaged over all pairs."""
    return np.mean(np.sum(unit(T) * unit(L), axis=1))

def mean_pairwise_cos(D):
    """Average cosine similarity between all pairs of offset vectors."""
    U = unit(D)
    sims = U @ U.T
    n = len(U)
    # the diagonal is each offset compared with itself (always 1), so leave it out
    off_diagonal = sims[~np.eye(n, dtype=bool)]
    return off_diagonal.mean()

def offsets(pairs):
    T = np.stack([kv_full[t] for t, _ in pairs])
    L = np.stack([kv_full[l] for _, l in pairs])
    return T, L, T - L

rng = np.random.default_rng(0) # rabdom number generator
all_pairs = [(t, l) for v in suffix_info.values() for t, l, _ in v]

rows = []
for s in suffixes:
    pairs = [(t, l) for t, l, _ in suffix_info[s]]
    T, L, D = offsets(pairs)
    # baseline: uses the same number of random pairs as the suffix has
    idx = rng.choice(len(all_pairs), size=len(pairs), replace=False)
    random_pairs = [all_pairs[i] for i in idx]
    _, _, D_rand = offsets(random_pairs)
    rows.append({
        "suffix": s, "n": len(pairs),
        "cos(token, lemma)": token_lemma_cos(T, L),
        "offset consistency": mean_pairwise_cos(D),
        "random baseline": mean_pairwise_cos(D_rand),
    })

res = pd.DataFrame(rows).round(3)
print(res.to_string(index=False))